# 01b v7 — Préparation des données

**Prérequis** : avoir exécuté `01a_clustering_v7.ipynb` jusqu'à la sauvegarde.

**Changements v7** :
- Modèle d'embedding : `all-MiniLM-L6-v2` (384d) → `cardiffnlp/twitter-roberta-base` (768d)
- `BATCH_SIZE_EMBED` : 512 → 256 (roberta-base plus lourd en mémoire)
- `EMBED_DIM` dynamique (768) — remplace les `384` hardcodés dans FAISS + reshape
- ⚠️ Après Bloc 4 : vérifier que les cluster IDs correspondent à `CLIENT_LABELS`

**Changements v6** :
- `tokenize` : fallback `[<UNK>]` si séquence vide après nettoyage (tweets non-latins)
- Cellule de diagnostic ajoutée après tokenisation

**Changements v5** :
- `CLIENT_LABELS` : 11 classes (suppression `other`)
- `COMPANY_LABELS` : 9 classes
- Bloc 13 ajouté : dataset de paires conversationnelles (client ↔ company)

## Bloc 1 : Config & imports

In [ ]:
import os, pickle, json, re
import numpy as np
import pandas as pd
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.cluster import MiniBatchKMeans
import faiss
import emoji
from sentence_transformers import SentenceTransformer

DATA_DIR             = '../data/'
VOCAB_SIZE           = 15000
MAX_LEN              = 64
BATCH_SIZE           = 64
K                    = 10
SIMILARITY_THRESHOLD = 0.85
BATCH_SIZE_EMBED = 256  
SAMPLE_CLUSTER       = 200000
N_MAX                = 100000
RANDOM_STATE         = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

## Bloc 2 : Chargement des données nettoyées (depuis 01a)

In [ ]:
with open(DATA_DIR + 'df_client.pkl', 'rb') as f:
    df_client = pickle.load(f)
with open(DATA_DIR + 'df_company.pkl', 'rb') as f:
    df_company = pickle.load(f)

print(f'df_client  : {len(df_client):,} tweets')
print(f'df_company : {len(df_company):,} tweets')

## Bloc 3 — Définition des classes

Dictionnaires validés sur la sortie de `01a_clustering_v3.ipynb` (K=12 client, K=9 company).

- **Client** : K=12 classes (exploration K=8→10→12→14, K=12 retenu)
- **Company** : K=9 classes (exploration K=7→8→9→11, K=9 retenu)

In [ ]:
# ── Flux client — mapping validé sur Bloc 4b (roberta-base, K=11) ────────────
# C6='short_ack' est supprimé en Bloc 6 → 10 classes finales (labels 0-9)
CLIENT_LABELS = {
    0:  'general_complaint',         # O2, AsurionCares, phone, fix, service
    1:  'food_delivery_complaint',   # Sainsbury's, delivery driver, order not delivered
    2:  'non_english',               # que/por/pas/ich/pour — tweets non-anglais
    3:  'account_issue',             # Spectrum, Chase, account, email, call, issue
    4:  'ios_device_issue',          # iphone, ios, app, update, fix — Apple/iOS
    5:  'billing_refund',            # order, amazon, customer, billing
    6:  'short_ack',                 # "Thanks!", "done", "Yep." — courts acks → supprimé
    7:  'service_complaint',         # fix, help, service, phone, AmericanAir
    8:  'transport_complaint',       # flight, train, delayed, British Airways
    9:  'tech_general',              # phone, Apple, card, iOS, fix
    10: 'no_response',               # customer service, account, OneDrive, Dropbox
}

# ── Flux company — 9 classes (inchangé) ──────────────────────────────────────
COMPANY_LABELS = {
    0: 'general_ack',
    1: 'order_apology_dm',
    2: 'redirect_dm',
    3: 'flight_apology',
    4: 'non_english',
    5: 'ios_dm_support',
    6: 'tech_support_dm',
    7: 'store_feedback_or_product_info',
    8: 'redirect_dm_help',
}

N_CLIENT  = len(CLIENT_LABELS)   # 11 (pour KMeans — réduit à 10 après Bloc 6)
N_COMPANY = len(COMPANY_LABELS)  # 9
print('Client  :', list(CLIENT_LABELS.values()))
print('Company :', list(COMPANY_LABELS.values()))
print(f'N_CLIENT={N_CLIENT} (KMeans) | N_COMPANY={N_COMPANY}')

## Bloc 4 : Labellisation KMeans (client + company)

In [ ]:
from sentence_transformers import models
word_embedding_model = models.Transformer(
    'cardiffnlp/twitter-roberta-base',
    max_seq_length=None,
    model_kwargs={'use_safetensors': True}
)
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
embed_model   = SentenceTransformer(modules=[word_embedding_model, pooling_model], device=str(device))
EMBED_DIM     = word_embedding_model.get_word_embedding_dimension()  
print(f'EMBED_DIM : {EMBED_DIM}')

#  Client 
df_sample_cl = df_client.sample(min(SAMPLE_CLUSTER, len(df_client)), random_state=RANDOM_STATE)
emb_sample_cl = embed_model.encode(
    df_sample_cl['text'].tolist(), batch_size=BATCH_SIZE_EMBED,
    show_progress_bar=True, convert_to_numpy=True
).astype(np.float32)

km_client = MiniBatchKMeans(n_clusters=N_CLIENT, random_state=RANDOM_STATE, n_init=15)
sample_labels_cl = km_client.fit_predict(emb_sample_cl)
df_labeled_client = df_sample_cl.reset_index(drop=True).copy()
df_labeled_client['label']      = sample_labels_cl
df_labeled_client['confidence'] = 1.0
sampled_idx_cl = set(df_sample_cl.index)
df_uncertain_client = df_client[~df_client.index.isin(sampled_idx_cl)].copy()
print(f'Client  — Labellisés : {len(df_labeled_client):,} | Incertains : {len(df_uncertain_client):,}')
print(df_labeled_client['label'].value_counts().sort_index())

#  Company 
df_sample_co = df_company.sample(min(SAMPLE_CLUSTER, len(df_company)), random_state=RANDOM_STATE)
emb_sample_co = embed_model.encode(
    df_sample_co['text'].tolist(), batch_size=BATCH_SIZE_EMBED,
    show_progress_bar=True, convert_to_numpy=True
).astype(np.float32)
km_company = MiniBatchKMeans(n_clusters=N_COMPANY, random_state=RANDOM_STATE, n_init=15)
sample_labels_co = km_company.fit_predict(emb_sample_co)
df_labeled_company = df_sample_co.reset_index(drop=True).copy()
df_labeled_company['label']      = sample_labels_co
df_labeled_company['confidence'] = 1.0
sampled_idx_co = set(df_sample_co.index)
df_uncertain_company = df_company[~df_company.index.isin(sampled_idx_co)].copy()
print(f'Company — Labellisés : {len(df_labeled_company):,} | Incertains : {len(df_uncertain_company):,}')
print(df_labeled_company['label'].value_counts().sort_index())

## Bloc 5 : Label Propagation (FAISS KNN × 2 flux)

In [ ]:
def label_propagation(df_labeled, df_uncertain, k=K, threshold=SIMILARITY_THRESHOLD):
    if len(df_labeled) == 0 or len(df_uncertain) == 0:
        print('  Skipped (labeled ou uncertain vide)')
        return pd.DataFrame()

    print(f'  Encodage {len(df_labeled):,} labellisés...')
    emb_labeled = embed_model.encode(
        df_labeled['text'].tolist(), batch_size=BATCH_SIZE_EMBED,
        show_progress_bar=True, convert_to_numpy=True
    ).astype(np.float32).reshape(-1, EMBED_DIM)
    faiss.normalize_L2(emb_labeled)

    print(f'  Encodage {len(df_uncertain):,} incertains...')
    emb_uncertain = embed_model.encode(
        df_uncertain['text'].tolist(), batch_size=BATCH_SIZE_EMBED,
        show_progress_bar=True, convert_to_numpy=True
    ).astype(np.float32).reshape(-1, EMBED_DIM)
    faiss.normalize_L2(emb_uncertain)

    index = faiss.IndexFlatIP(EMBED_DIM)
    index.add(emb_labeled)
    sims, indices = index.search(emb_uncertain, k)

    labels_labeled = np.array(df_labeled['label'].tolist(), dtype=int)

    valid_mask = (sims >= threshold) & (indices >= 0)
    has_valid  = valid_mask.any(axis=1)
    valid_rows = np.where(has_valid)[0]

    propagated_labels = np.full(len(df_uncertain), -1, dtype=int)
    propagated_confs  = np.zeros(len(df_uncertain))

    for i in valid_rows:
        neighbor_labels = labels_labeled[indices[i][valid_mask[i]]]
        propagated_labels[i] = Counter(neighbor_labels).most_common(1)[0][0]
        propagated_confs[i]  = sims[i][valid_mask[i]].mean()

    df_prop = df_uncertain.iloc[valid_rows].copy()
    df_prop['label']      = propagated_labels[valid_rows]
    df_prop['confidence'] = propagated_confs[valid_rows]

    print(f'  Propagés : {len(df_prop):,} | Rejetés : {(~has_valid).sum():,}')
    return df_prop

print('=== Label Propagation — Client ===')
df_prop_client = label_propagation(df_labeled_client, df_uncertain_client)
df_full_client = pd.concat([df_labeled_client, df_prop_client], ignore_index=True)
print(f'Dataset client total : {len(df_full_client):,}')

print('\n=== Label Propagation — Entreprise ===')
df_prop_company = label_propagation(df_labeled_company, df_uncertain_company)
df_full_company = pd.concat([df_labeled_company, df_prop_company], ignore_index=True)
print(f'Dataset entreprise total : {len(df_full_company):,}')

## Bloc 6 : Équilibrage (N_MAX par classe)

In [ ]:
def balance_dataset(df, name, label_col='label', n_max=N_MAX):
    if len(df) == 0:
        print(f'{name} : dataset vide, skipped')
        return df
    groups = []
    for lbl, grp in df.groupby(label_col):
        groups.append(grp.sample(min(len(grp), n_max), random_state=RANDOM_STATE))
    out = pd.concat(groups).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'{name} — distribution après équilibrage :')
    counts = out[label_col].value_counts().sort_index()
    label_map = CLIENT_LABELS if name == 'Client' else COMPANY_LABELS
    for idx, cnt in counts.items():
        print(f'  {idx:2d} — {label_map.get(idx, "?"):35s} : {cnt:,}')
    return out

df_bal_client  = balance_dataset(df_full_client,  'Client')

# ── Suppression de 'short_ack' (cluster 6) ───────────────────────────────────
# C6 = courts remerciements ("Thanks!", "done") sans contenu sémantique propre
# Même comportement que l'ancien 'other' → cause confusion chez DistilBERT
# Remap : 7→6, 8→7, 9→8, 10→9 — labels finaux continus 0-9 (10 classes)
df_bal_client  = df_bal_client[df_bal_client['label'] != 6].reset_index(drop=True)
label_remap    = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5, 7:6, 8:7, 9:8, 10:9}
df_bal_client['label'] = df_bal_client['label'].map(label_remap)

# Mise à jour CLIENT_LABELS avec les 10 classes finales (IDs 0-9 après remap)
CLIENT_LABELS = {
    0: 'general_complaint',
    1: 'food_delivery_complaint',
    2: 'non_english',
    3: 'account_issue',
    4: 'ios_device_issue',
    5: 'billing_refund',
    6: 'service_complaint',      
    7: 'transport_complaint',   
    8: 'tech_general',          
    9: 'no_response',           
}
N_CLIENT = len(CLIENT_LABELS)  # 10
print(f'\nAprès suppression "short_ack" : {len(df_bal_client):,} tweets client, {N_CLIENT} classes')

df_bal_company = balance_dataset(df_full_company, 'Entreprise')

## Bloc 7 : Split stratifié 70/15/15

In [ ]:
def stratified_split(df, name, label_col='label'):
    if len(df) == 0:
        print(f'{name} : dataset vide, split ignoré')
        empty = np.array([])
        return (empty, empty), (empty, empty), (empty, empty)
    X = np.array(df['text'].tolist())
    y = np.array(df[label_col].tolist(), dtype=int)
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
    X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE)
    print(f'{name} — Train:{len(X_tr):,} Val:{len(X_val):,} Test:{len(X_te):,}')
    return (X_tr, y_tr), (X_val, y_val), (X_te, y_te)

client_train,  client_val,  client_test  = stratified_split(df_bal_client,  'Client')
company_train, company_val, company_test = stratified_split(df_bal_company, 'Entreprise')

## Bloc 8 : Nettoyage textuel

In [ ]:
def clean_text(text):
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = emoji.demojize(text, delimiters=(' ', ' '))
    text = re.sub(r'[^a-zA-Z0-9\s_:]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def clean_split(split_tuple):
    X, y = split_tuple
    return np.array([clean_text(t) for t in X]), y

client_train  = clean_split(client_train)
client_val    = clean_split(client_val)
client_test   = clean_split(client_test)
company_train = clean_split(company_train)
company_val   = clean_split(company_val)
company_test  = clean_split(company_test)

print('Exemple nettoyé :', client_train[0][0])

## Bloc 9 : Vocabulaire (pattern TP5 — train client uniquement)

In [ ]:
counter = Counter()
for text in client_train[0]:
    counter.update(text.split())

vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

PAD_IDX = vocab['<PAD>']
print(f'Taille vocabulaire : {len(vocab):,}')
print(f'Top 10 : {counter.most_common(10)}')

## Bloc 10 : Tokenisation (pattern TP5)

In [ ]:
def tokenize(text, vocab, max_len=MAX_LEN):
    tokens = text.split()[:max_len]
    ids = [vocab.get(w, vocab['<UNK>']) for w in tokens]
    return ids if ids else [vocab['<UNK>']]  # tweet non-latin nettoyé à "" → un seul <UNK>

def encode_split(split_tuple):
    X, y = split_tuple
    return [(tokenize(t, vocab), int(label)) for t, label in zip(X, y)]

client_train_enc  = encode_split(client_train)
client_val_enc    = encode_split(client_val)
client_test_enc   = encode_split(client_test)
company_train_enc = encode_split(company_train)
company_val_enc   = encode_split(company_val)
company_test_enc  = encode_split(company_test)

print(f'Exemple : {client_train_enc[0][0][:10]} → label {client_train_enc[0][1]}')

In [ ]:
for split_name, enc in [('client_train',  client_train_enc),
                        ('client_val',    client_val_enc),
                        ('client_test',   client_test_enc),
                        ('company_train', company_train_enc),
                        ('company_val',   company_val_enc),
                        ('company_test',  company_test_enc)]:
    single = [(t, l) for t, l in enc if len(t) == 1]
    short  = [(t, l) for t, l in enc if 2 <= len(t) <= 3]
    trunc  = [(t, l) for t, l in enc if len(t) == MAX_LEN]

    print(f'{split_name:20s} total={len(enc):,} | '
          f'single_unk={len(single):4d} | short(2-3)={len(short):5d} | tronqués={len(trunc):4d}')

    if single:
        from collections import Counter
        cls = Counter(l for _, l in single)
        # Utilise CLIENT_LABELS ou COMPANY_LABELS en fonction du split
        label_map = CLIENT_LABELS if 'client' in split_name else COMPANY_LABELS
        print(f'  → classes single_unk : { {label_map.get(k, k): v for k, v in cls.items()} }')

## Bloc 11 : Dataset + DataLoader (pattern TP5)

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, encoded_data):
        self.texts  = [d[0] for d in encoded_data]
        self.labels = [d[1] for d in encoded_data]
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(
        [torch.tensor(t, dtype=torch.long) for t in texts],
        batch_first=True, padding_value=PAD_IDX
    )
    return texts_padded, torch.tensor(labels, dtype=torch.long)

def make_loaders(train_enc, val_enc, test_enc):
    if len(train_enc) == 0:
        print('  Warning: dataset vide, loaders non créés')
        return None
    return {
        'train': DataLoader(IntentDataset(train_enc), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn),
        'val'  : DataLoader(IntentDataset(val_enc),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn),
        'test' : DataLoader(IntentDataset(test_enc),  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn),
    }

client_loaders  = make_loaders(client_train_enc,  client_val_enc,  client_test_enc)
company_loaders = make_loaders(company_train_enc, company_val_enc, company_test_enc)

if client_loaders:
    for texts, labels in client_loaders['train']:
        print(f'Batch client  — textes: {texts.shape}, labels: {labels.shape}')
        break
if company_loaders:
    for texts, labels in company_loaders['train']:
        print(f'Batch company — textes: {texts.shape}, labels: {labels.shape}')
        break

## Bloc 12 : Sauvegarde des artefacts

In [ ]:
with open(DATA_DIR + 'vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)

# Convertir les clés int en str pour JSON
label_maps = {
    'client' : {str(k): v for k, v in CLIENT_LABELS.items()},
    'company': {str(k): v for k, v in COMPANY_LABELS.items()},
}
with open(DATA_DIR + 'label_maps.json', 'w') as f:
    json.dump(label_maps, f, indent=2)

with open(DATA_DIR + 'client_data.pkl', 'wb') as f:
    pickle.dump({'train': client_train_enc, 'val': client_val_enc, 'test': client_test_enc}, f)

with open(DATA_DIR + 'company_data.pkl', 'wb') as f:
    pickle.dump({'train': company_train_enc, 'val': company_val_enc, 'test': company_test_enc}, f)

print('Artefacts sauvegardés dans', DATA_DIR)
print(f'  vocab.pkl        : {len(vocab):,} tokens')
print(f'  label_maps.json  : client={N_CLIENT} classes, company={N_COMPANY} classes')
print(f'  client_data.pkl  : train/val/test')
print(f'  company_data.pkl : train/val/test')
print()
print('=== Rappel OUTPUT_DIM pour les notebooks modèles ===')
print(f'  Flux client  → OUTPUT_DIM = {N_CLIENT}')
print(f'  Flux company → OUTPUT_DIM = {N_COMPANY}')

In [ ]:
# ── Textes nettoyés pour DistilBERT (notre vocab custom est inutile pour lui) ─
# DistilBERT utilise son propre tokenizer HuggingFace → on sauvegarde les strings
client_text_data = {
    'train': [(text, int(label)) for text, label in zip(client_train[0], client_train[1])],
    'val':   [(text, int(label)) for text, label in zip(client_val[0],   client_val[1])],
    'test':  [(text, int(label)) for text, label in zip(client_test[0],  client_test[1])],
}
with open(DATA_DIR + 'client_text_data.pkl', 'wb') as f:
    pickle.dump(client_text_data, f)

print(f'  client_text_data.pkl : {len(client_text_data["train"]):,} train | '
      f'{len(client_text_data["val"]):,} val | {len(client_text_data["test"]):,} test')
print(f'  Exemple : "{client_text_data["train"][0][0][:60]}..." → label {client_text_data["train"][0][1]}')

## Bloc 13 : Dataset paires (client ↔ company)

Jointure conversationnelle : chaque tweet entreprise référence via `in_response_to_tweet_id`
le tweet client auquel il répond. On exploite ce lien pour créer un dataset de paires
`(text_client, text_company, label_client, label_company)` utilisé pour le modèle multi-task.

- **Inner join** : seuls les tweets mutuellement labellisés sont retenus
- Les datasets séparés `client_data.pkl` / `company_data.pkl` ne sont pas modifiés

In [ ]:
# ── Jointure conversationnelle ────────────────────────────────────────────────
# df_full_company.in_response_to_tweet_id → df_full_client.tweet_id

df_co_linked = df_full_company[
    ['text', 'label', 'in_response_to_tweet_id']
].dropna(subset=['in_response_to_tweet_id']).copy()

df_cl_lookup = df_full_client[['tweet_id', 'text', 'label']].rename(
    columns={'text': 'text_client', 'label': 'label_client'}
)

df_pairs = df_co_linked.merge(
    df_cl_lookup,
    left_on='in_response_to_tweet_id',
    right_on='tweet_id',
    how='inner'
).rename(columns={
    'text':  'text_company',
    'label': 'label_company'
})[['text_client', 'label_client', 'text_company', 'label_company']].reset_index(drop=True)

# Forcer int (le merge peut introduire des floats si certaines clés sont manquantes)
df_pairs['label_client']  = df_pairs['label_client'].astype(int)
df_pairs['label_company'] = df_pairs['label_company'].astype(int)

print(f'Paires trouvées : {len(df_pairs):,}')
print(f'\nDistribution label_client dans les paires :')
for idx, cnt in df_pairs['label_client'].value_counts().sort_index().items():
    print(f'  {idx:2d} — {CLIENT_LABELS[idx]:30s} : {cnt:,}')
print(f'\nDistribution label_company dans les paires :')
for idx, cnt in df_pairs['label_company'].value_counts().sort_index().items():
    print(f'  {idx:2d} — {COMPANY_LABELS[idx]:35s} : {cnt:,}')

# ── Split stratifié sur label_client ─────────────────────────────────────────
y_cl    = df_pairs['label_client'].values
idx_all = np.arange(len(df_pairs))

idx_tr,  idx_tmp = train_test_split(idx_all, test_size=0.30, stratify=y_cl,          random_state=RANDOM_STATE)
idx_val, idx_te  = train_test_split(idx_tmp, test_size=0.50, stratify=y_cl[idx_tmp], random_state=RANDOM_STATE)

# ── Tokenisation des deux côtés ───────────────────────────────────────────────
def encode_pairs_split(idx):
    rows = df_pairs.iloc[idx]
    result = []
    for _, row in rows.iterrows():
        tc  = tokenize(clean_text(row['text_client']),  vocab)
        tco = tokenize(clean_text(row['text_company']), vocab)
        result.append((tc, tco, int(row['label_client']), int(row['label_company'])))
    return result

pairs_train = encode_pairs_split(idx_tr)
pairs_val   = encode_pairs_split(idx_val)
pairs_test  = encode_pairs_split(idx_te)

print(f'\nSplit paires — Train:{len(pairs_train):,}  Val:{len(pairs_val):,}  Test:{len(pairs_test):,}')
print('Format : (tokens_client, tokens_company, label_client, label_company)')

# ── Sauvegarde ────────────────────────────────────────────────────────────────
with open(DATA_DIR + 'paired_data.pkl', 'wb') as f:
    pickle.dump({'train': pairs_train, 'val': pairs_val, 'test': pairs_test}, f)

print(f'\npaired_data.pkl sauvegardé dans {DATA_DIR}')